### **Prerequisites for the Lab**

Before the session, ensure you have the environment set up.

1. Install [Ollama](https://ollama.com/) and download a lightweight model: `ollama run llama3` (or `llama3.1` or `mistral`).
2. Install the required Python libraries:

In [59]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [60]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [61]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Open a log file to catch the output so it doesn't fill the OS pipe
log_file = open('/content/ollama_server.log', 'w')

subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

time.sleep(5)

In [62]:
!ollama pull gemma4:latest

In [63]:
!ollama pull nomic-embed-text

In [64]:
!pip install langchain langchain-ollama langchain-community langchain-chroma sqlalchemy chromadb faker

In [65]:
import json
from langchain_ollama import ChatOllama
from langchain_classic.agents import initialize_agent, AgentType
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.tools import tool

In [66]:
# 0. INITIALIZATION
print("[SYSTEM] Booting Customer Support Agent Engine...")
# Using temperature 0 for predictable, policy-compliant support interactions
llm = ChatOllama(model="gemma4:latest", base_url="http://localhost:11434", temperature=0)

[SYSTEM] Booting Customer Support Agent Engine...


In [67]:
import json
from sqlalchemy.ext.declarative import declarative_base
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
import random
from faker import Faker
from sqlalchemy import create_engine, Column, Integer, String, Text, ForeignKey, event
from sqlalchemy.orm import declarative_base, sessionmaker

In [82]:
# 1. SETUP SQL CRM

# Initialize Faker
fake = Faker()

# Database Setup
Base = declarative_base()
class Customer(Base):
    __tablename__ = 'customers'
    id = Column(Integer, primary_key=True)
    email = Column(String, unique=True)
    profile_summary = Column(Text)

class Order(Base):
    __tablename__ = 'orders'
    id = Column(String, primary_key=True)  # e.g., "ORD-9921"
    customer_email = Column(String, ForeignKey('customers.email'))
    item_name = Column(String)
    status = Column(String)

In [83]:
# 2. Delete the database file so we start from zero
if os.path.exists("enterprise_crm.db"):
    os.remove("enterprise_crm.db")
    print("Database wiped successfully.")

# 3. Re-initialize the engine and table
engine = create_engine("sqlite:///enterprise_crm.db")
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

In [84]:
# 1. Rollback the broken session first
session.rollback()

# 2. Update the generation function to use merge()
def generate_synthetic_data(n=20):
    for _ in range(n):
        name = fake.name()
        email = fake.unique.email()
        issues = ["frequent returns", "billing inquiry", "shipping delay", "positive feedback"]
        summary = f"{name}: High-value account. Recent history includes: {random.choice(issues)}."

        # USE MERGE INSTEAD OF ADD
        customer = Customer(email=email, profile_summary=summary)
        session.merge(customer)

    session.commit()

# Then generate your random data
generate_synthetic_data(20)

In [70]:
# 2. SETUP VECTOR STORE
vector_db = Chroma(
    persist_directory="./chroma_db",
    embedding_function=OllamaEmbeddings(model="nomic-embed-text")
)

In [71]:
# 3. REAL-TIME SYNC MECHANISM (SQL Trigger)
def sync_to_vector_db(mapper, connection, target):
    """Automatically update Vector DB when SQL record changes."""
    doc = Document(
        page_content=target.profile_summary,
        metadata={"email": target.email}
    )
    vector_db.add_documents([doc], ids=[target.email])
    print(f"[SYNC] Vector DB updated for: {target.email}")

event.listen(Customer, 'after_insert', sync_to_vector_db)
event.listen(Customer, 'after_update', sync_to_vector_db)

In [85]:
# 3. Seed Alice FRESH (No merge conflicts now)
try:
    alice = Customer(email="alice@email.com", profile_summary="Alice Smith: High-value, frequent returns.")
    order = Order(id="ORD-9921", customer_email="alice@email.com",
                  item_name="Wireless Headphones", status="Delayed in Transit")
    session.add(alice)
    session.add(order)
    session.commit()
    print("Seeding successful!")
except Exception as e:
    session.rollback()
    print(f"Critical failure: {e}")

Seeding successful!


In [74]:
# The Agent can now call search_crm_memory("Alice Smith") to get the data
print(search_crm_memory.invoke("Alice Smith"))

Alice Smith: High-value, frequent returns.


In [75]:
from langchain_core.tools import tool

@tool
def search_crm_memory(query: str):
    """Searches the CRM vector database for customer context. Use this first if you only have a name or partial info."""
    retriever = vector_db.as_retriever(search_kwargs={"k": 2})
    results = retriever.invoke(query)
    return "\n".join([res.page_content for res in results])

@tool
def fetch_customer_profile(email: str):
    """Fetches full customer profile from the SQL CRM database using an email address."""
    customer = session.query(Customer).filter(Customer.email == email).first()
    if customer:
        return f"Name: {customer.email.split('@')[0]}, Data: {customer.profile_summary}"
    return "Customer not found."

@tool
def lookup_order_status(email: str):
    """
    Looks up the status of a customer's order.
    Input MUST be the customer's email address.
    Use this tool to find order details. Do not ask the user for an Order ID.
    """
    order = session.query(Order).filter(Order.customer_email == email).first()
    if order:
        return f"Item: {order.item_name}, Status: {order.status}"
    return "No active orders found for this email."

@tool
def process_refund(order_id: str):
    """Cancels an order and processes a refund. Only use this if the customer explicitly requests it."""
    return f"Success: Refund processed for order {order_id}."

In [76]:
# 3. SHORT-TERM MEMORY (STM) CONFIGURATION

# ConversationBufferMemory stores the immediate back-and-forth chat history.
# 'memory_key="chat_history"' aligns with LangChain's conversational agent prompt.
short_term_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

In [77]:
# Define tools list
tools = [search_crm_memory, fetch_customer_profile, lookup_order_status, process_refund]

# System prompt: Explicitly tell the agent to use tools
agent_kwargs = {
    "prefix": """You are an expert Customer Support Agent.
    Your goal is to assist customers using the provided tools.
    1. If you have the customer's email, use 'fetch_customer_profile' or 'lookup_order_status' immediately.
    2. NEVER ask the user for an Order ID if you can retrieve it via your tools using their email.
    3. Be empathetic, concise, and helpful."""
}

# Initialize Agent
support_agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=short_term_memory,
    agent_kwargs=agent_kwargs,
    verbose=True,
    handle_parsing_errors=True
)

In [78]:
# 5. MULTI-TURN EXECUTION (Demonstrating STM, LTM, and Feedback Loops)

def simulate_conversation(user_messages):
    print("INITIATING CUSTOMER SUPPORT CHAT SESSION")

    for idx, msg in enumerate(user_messages, 1):
        print(f"\nUSER (Turn {idx}): {msg}")
        # The agent processes the input, uses STM (memory), and decides on tools
        response = support_agent.run(input=msg)
        print(f"\nAGENT: {response}")

In [88]:
# Turn 1
response1 = support_agent.run("Hi, I'm Alice. My email is alice@email.com. Can you help me with my order?")
print(f"AGENT: {response1}")



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: lookup_order_status
Action Input: alice@email.com
Observation: Item: Wireless Headphones, Status: Delayed in Transit
Thought:Thought: Do I need to use a tool? No
AI: Hello Alice! Thank you for reaching out again. I've checked on your order using your email, and I can confirm that your **Wireless Headphones** are currently marked as **Delayed in Transit**.

I sincerely apologize for this unexpected delay. We are actively monitoring the shipment and will provide an update as soon as we have more information regarding its expected delivery date.

Please let me know if you would like me to check on any other details, or if you would prefer to proceed with a cancellation and refund instead!

> Finished chain.
AGENT: Hello Alice! Thank you for reaching out again. I've checked on your order using your email, and I can confirm that your **Wireless Headphones** are currently marked as **Delayed in Transit**.



In [89]:
# Turn 2 (The agent remembers Alice's email from Turn 1)
response2 = support_agent.run("I want to know where my active order is.")
print(f"AGENT: {response2}")



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: lookup_order_status
Action Input: alice@email.com
Observation: Item: Wireless Headphones, Status: Delayed in Transit
Thought:Thought: Do I need to use a tool? No
AI: Hello Alice! Thank you for checking in again. I understand you want an update on your order, and I'm happy to provide that for you.

I have checked our system, and I can confirm that your **Wireless Headphones** are currently marked as **Delayed in Transit**. We sincerely apologize for this unexpected delay and any frustration it has caused.

We are actively monitoring the shipment and will provide an update on its expected delivery date as soon as we receive more information from the carrier.

Please let me know if you would like me to proceed with a cancellation and refund instead, or if there is anything else I can assist you with today!

> Finished chain.
AGENT: Hello Alice! Thank you for checking in again. I understand you want an up

In [90]:
# Turn 3 (The agent remembers the context of the order from Turn 2)
response3 = support_agent.run("This is taking too long. Please just cancel and refund it.")
print(f"AGENT: {response3}")



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: lookup_order_status
Action Input: alice@email.com
Observation: Item: Wireless Headphones, Status: Delayed in Transit
Thought:Thought: Do I need to use a tool? Yes
Action: process_refund
Action Input: alice@email.com
Observation: Success: Refund processed for order alice@email.com.
Thought:Thought: Do I need to use a tool? No
AI: I completely understand your frustration with the delay, and I sincerely apologize that this process has been so difficult.

I have successfully processed the cancellation and full refund for your Wireless Headphones order. You should see the credit reflected on your original payment method within 3-5 business days, depending on your bank.

Please let me know if there is anything else I can assist you with today!

> Finished chain.
AGENT: I completely understand your frustration with the delay, and I sincerely apologize that this process has been so difficult.

I have successf